# 🤖 Step 4: Enhanced AI Detection Training v2
## IntegriCheck — Academic Integrity Platform

### 🎯 Is Notebook Ka Kaam Kya Hai?
Yeh notebook **AI-generated text detect karne ke liye ensemble ML model** train karta hai.

### 📊 v1 vs v2 Upgrades
| Component | v1 | v2 |
|-----------|----|----|
| Features | 12 | **18** |
| Model | Random Forest only | **Ensemble (RF + GB + LR)** |
| Probabilities | Raw | **Calibrated (Platt scaling)** |
| Training data | ~20k | **~50k+** |
| Target accuracy | 88.9% | **92%+** |

### 🆕 New Features (v2):
`discourse_marker_rate`, `passive_voice_ratio`, `hapax_ratio`, `para_length_std`, `sent_overlap`, `colon_ratio`, `max_sent_len`, `min_sent_len`

### ⏱️ Runtime (CPU): ~10-20 minutes

## Cell 1: Setup + Dataset Load
**Kya karta hai:** Libraries import karo aur AI detection dataset load karo.

### Dataset Check:
- `label=0` → Human written samples
- `label=1` → AI generated samples
- Balance check important hai — zyada imbalanced data → biased model

In [2]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
import joblib
import re
import matplotlib
matplotlib.use('Agg')   # Server/notebook mein interactive window nahi chahiye
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import Counter
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── Project root (same consistent structure) ─────────────────────────────────
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)

BASE_DIR = PROJECT_ROOT

# ── Paths ─────────────────────────────────────────────────────────────────────
MODELS_DIR    = os.path.join(PROJECT_ROOT, 'data', 'models')
PROCESSED_DIR  = os.path.join(PROJECT_ROOT, 'data', 'processed')

os.makedirs(MODELS_DIR, exist_ok=True)

# ── Load dataset ─────────────────────────────────────────────────────────────
ai_path = os.path.join(PROCESSED_DIR, 'ai_detection_data.csv')

if not os.path.exists(ai_path):
    raise FileNotFoundError(
        "❌ ai_detection_data.csv nahi mila!\n"
        "   Pehle step2_data_collection_v2.ipynb chalaao."
    )

df = pd.read_csv(ai_path)
df = df.dropna(subset=['text', 'label'])
df['text']  = df['text'].astype(str)
df['label'] = df['label'].astype(int)

# ── Print header ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  IntegriCheck — AI Detection Training v2")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# ── Dataset overview ─────────────────────────────────────────────────────────
print("\nProject paths:")
print(f"  Root      : {PROJECT_ROOT}")
print(f"  Models    : {MODELS_DIR}")
print(f"  Processed : {PROCESSED_DIR}")

print(f"\n📊 Dataset Overview:")
print(f"  Total samples:   {len(df):,}")
print(f"  Human (0):       {len(df[df.label==0]):,} ({len(df[df.label==0])/len(df)*100:.1f}%)")
print(f"  AI (1):          {len(df[df.label==1]):,} ({len(df[df.label==1])/len(df)*100:.1f}%)")

if 'source' in df.columns:
    print(f"\n  Sources:")
    for src, cnt in df['source'].value_counts().head(8).items():
        print(f"    {src:35s}: {cnt:,}")

  IntegriCheck — AI Detection Training v2
  Started: 2026-05-11 18:51:37

Project paths:
  Root      : e:\ADD\integricheck_project
  Models    : e:\ADD\integricheck_project\data\models
  Processed : e:\ADD\integricheck_project\data\processed

📊 Dataset Overview:
  Total samples:   50,160
  Human (0):       25,080 (50.0%)
  AI (1):          25,080 (50.0%)

  Sources:
    gptwiki_ai                         : 15,094
    gptwiki_human                      : 10,683
    research_ai                        : 9,986
    writingprompts_human               : 7,231
    research_human                     : 7,166


## Cell 2: Stopwords + AI Discourse Markers Define Karo
**Kya karta hai:** Feature extraction ke liye required word lists define karta hai.

### Kyun Yeh Lists?

**Stopwords:** "the", "a", "is" jaise common words — yeh AI vs Human distinction nahi dete, so feature calculation mein exclude karo.

**AI Discourse Markers:** Yeh words/phrases AI text mein bahut common hain:
- "Furthermore", "Moreover", "Additionally" — AI transition words
- "It is important to note" — AI signature phrase
- "Delve", "Intricate", "Pivotal" — overused AI vocabulary
- "Seamlessly", "Robust", "Holistic" — AI buzzwords

**Research finding:** GPTZero aur similar tools bhi discourse markers ko primary signal maante hain.

In [3]:
# ── Stopwords (basic English) ──────────────────────────────────────────────────
STOP_WORDS = set([
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','being','have','has','had','do',
    'does','did','will','would','could','should','may','might','shall',
    'this','that','these','those','it','its','i','we','you','he','she','they',
    'my','your','our','his','her','their','what','which','who','how','when',
    'where','why','not','no','so','if','as','by','from','into','than','then',
    'very','just','also','more','most','some','any','all','both','each',
    'few','many','much','other','such','only','own','same','too','well',
    'even','still','back','way','first','last','long','great','little'
])

# ── AI-specific discourse markers (strong AI signal) ──────────────────────────
AI_DISCOURSE = [
    # Transition words (AI overuses these)
    'furthermore', 'moreover', 'additionally', 'consequently', 'therefore',
    # Summary phrases (very common in AI essays)
    'in conclusion', 'to summarize', 'in summary',
    # Filler phrases (AI signature patterns)
    'it is important to note', 'it is worth noting', 'needless to say',
    'it goes without saying', 'that being said', 'having said that',
    'on the other hand', 'in other words', 'with that in mind',
    # Passive analysis phrases
    'this highlights', 'this underscores', 'this demonstrates',
    'in this context', 'to this end',
    # Overused AI vocabulary (GPTisms)
    'delve', 'intricate', 'pivotal', 'crucial', 'harness', 'leverage',
    'seamlessly', 'multifaceted', 'embark', 'fostering', 'landscape',
    'realm', 'robust', 'holistic', 'nuanced', 'paradigm', 'synergy',
]

print(f"✅ Stopwords defined: {len(STOP_WORDS)} words")
print(f"✅ AI discourse markers: {len(AI_DISCOURSE)} patterns")
print(f"\n  Sample AI markers: {AI_DISCOURSE[:5]}")
print(f"  ...and GPTisms: {AI_DISCOURSE[-5:]}")


✅ Stopwords defined: 97 words
✅ AI discourse markers: 39 patterns

  Sample AI markers: ['furthermore', 'moreover', 'additionally', 'consequently', 'therefore']
  ...and GPTisms: ['robust', 'holistic', 'nuanced', 'paradigm', 'synergy']


## Cell 3: 18-Feature Extraction Function
**Kya karta hai:** Har text se 18 numerical features extract karta hai — yeh features AI vs Human text mein statistically different hote hain.

### Feature Groups (18 total):

| # | Feature | AI Tendency | Why? |
|---|---------|-------------|------|
| 0 | avg_sent_len | HIGHER | AI writes longer sentences |
| 1 | sent_len_std | LOWER | AI sentences more consistent |
| 2 | burstiness | MORE NEGATIVE | AI: very uniform lengths |
| 3 | max_sent_len | HIGHER | AI has very long sentences |
| 4 | min_sent_len | HIGHER | AI rarely writes short sentences |
| 5 | vocab_richness | HIGHER | AI uses diverse words |
| 6 | hapax_ratio | LOWER | AI repeats words more |
| 7 | avg_word_len | HIGHER | AI uses longer/complex words |
| 8 | stop_ratio | LOWER | AI uses fewer filler words |
| 9 | avg_word_freq | LOWER | AI distributes word frequency evenly |
| 10 | punct_ratio | HIGHER | AI uses more punctuation |
| 11 | comma_ratio | HIGHER | AI uses many commas |
| 12 | period_ratio | — | Varies by model |
| 13 | colon_ratio | HIGHER | AI loves bullet points via colons |
| 14 | discourse_marker_rate | MUCH HIGHER | "Furthermore", "Moreover" = AI |
| 15 | passive_voice_ratio | HIGHER | AI uses more passive voice |
| 16 | para_length_std | LOWER | AI paragraphs very uniform |
| 17 | sent_overlap | HIGHER | AI transitions smoothly |

In [4]:
def sent_tokenize_simple(text):
    """Simple sentence tokenizer — NLTK preferred, regex fallback."""
    try:
        import nltk
        return nltk.sent_tokenize(text)
    except Exception:
        sents = re.split(r'(?<=[.!?])\s+', text)
        return [s.strip() for s in sents if s.strip()]


def extract_features_v2(text: str) -> list:
    """
    18 features extract karo AI detection ke liye.
    
    Returns: List of 18 float values
    """
    if not text or len(text) < 20:
        return [0.0] * 18

    # ── Tokenize ──────────────────────────────────────────────────────────────
    sents = sent_tokenize_simple(text)
    sents = [s.strip() for s in sents if len(s.strip()) > 5]
    words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
    chars = len(text)

    if not sents or not words:
        return [0.0] * 18

    # ── Group 1: Sentence-level stats (features 0-4) ──────────────────────────
    sent_lens  = [len(s.split()) for s in sents]
    mean_sl    = float(np.mean(sent_lens))          # Avg sentence length
    std_sl     = float(np.std(sent_lens))           # Sentence length variation
    # Burstiness: negative = very consistent (AI), positive = irregular (human)
    burstiness = (std_sl - mean_sl) / (std_sl + mean_sl + 1e-9)
    max_sl     = float(max(sent_lens))              # Longest sentence
    min_sl     = float(min(sent_lens))              # Shortest sentence

    # ── Group 2: Vocabulary (features 5-9) ───────────────────────────────────
    wc       = Counter(words)
    vocab_r  = len(set(words)) / (len(words) + 1)  # Unique words ratio
    # Hapax = words appearing exactly once (human writers more unique)
    hapax_r  = sum(1 for w, c in wc.items() if c == 1) / (len(wc) + 1)
    avg_wl   = float(np.mean([len(w) for w in words]))
    stop_r   = sum(1 for w in words if w in STOP_WORDS) / (len(words) + 1)
    avg_wf   = float(np.mean(list(wc.values())))   # Avg word frequency

    # ── Group 3: Punctuation (features 10-13) ─────────────────────────────────
    punct_r  = sum(1 for c in text if c in '.,;:!?') / (chars + 1)
    comma_r  = text.count(',') / (chars + 1)
    period_r = text.count('.') / (chars + 1)
    colon_r  = text.count(':') / (chars + 1)   # AI uses colons for lists

    # ── Group 4: AI Discourse Markers (features 14-15) ────────────────────────
    text_l   = text.lower()
    dm_count = sum(1 for dm in AI_DISCOURSE if dm in text_l)
    dm_rate  = dm_count / (len(sents) + 1)     # Discourse markers per sentence

    # Passive voice: "was written", "is known", "were analyzed"
    passive  = re.findall(r'\b(is|are|was|were|be|been|being)\s+\w+ed\b', text_l)
    passive_r = len(passive) / (len(sents) + 1)

    # ── Group 5: Paragraph Coherence (features 16-17) ─────────────────────────
    paragraphs = [p.strip() for p in text.split('\n\n') if len(p.strip()) > 20]

    if len(paragraphs) > 1:
        para_lens = [len(p.split()) for p in paragraphs]
        # Low para_std = very uniform = AI signature
        para_std  = float(np.std(para_lens)) / (np.mean(para_lens) + 1)
    else:
        para_std = 0.0

    # Sentence-to-sentence lexical overlap (AI transitions smoothly)
    if len(sents) >= 3:
        overlaps = []
        for i in range(len(sents) - 1):
            w1 = set(re.findall(r'\b[a-z]+\b', sents[i].lower()))
            w2 = set(re.findall(r'\b[a-z]+\b', sents[i+1].lower()))
            if w1 and w2:
                overlaps.append(len(w1 & w2) / len(w1 | w2))
        avg_overlap = float(np.mean(overlaps)) if overlaps else 0.0
    else:
        avg_overlap = 0.0

    return [
        mean_sl, std_sl, burstiness, max_sl, min_sl,      # 0-4  Sentence stats
        vocab_r, hapax_r, avg_wl, stop_r, avg_wf,         # 5-9  Vocabulary
        punct_r, comma_r, period_r, colon_r,               # 10-13 Punctuation
        dm_rate, passive_r,                                # 14-15 AI markers
        para_std, avg_overlap,                             # 16-17 Coherence
    ]


FEATURE_NAMES = [
    'avg_sent_len', 'sent_len_std', 'burstiness', 'max_sent_len', 'min_sent_len',
    'vocab_richness', 'hapax_ratio', 'avg_word_len', 'stop_ratio', 'avg_word_freq',
    'punct_ratio', 'comma_ratio', 'period_ratio', 'colon_ratio',
    'discourse_marker_rate', 'passive_voice_ratio',
    'para_length_std', 'sent_overlap',
]

# Quick test
sample_human = "I went to the market today. It was very crowded. I bought some vegetables."
sample_ai    = "Furthermore, it is important to note that machine learning leverages robust algorithms to seamlessly process data. Moreover, the intricate nature of these paradigms necessitates holistic approaches."

print("Feature extraction test:")
print(f"  Sample human text features: {extract_features_v2(sample_human)[:5]}")
print(f"  Sample AI text features:    {extract_features_v2(sample_ai)[:5]}")
print(f"\n✅ Feature extractor ready! ({len(FEATURE_NAMES)} features)")


Feature extraction test:
  Sample human text features: [4.666666666666667, 0.9428090415820634, -0.6638512791747759, 6.0, 4.0]
  Sample AI text features:    [13.0, 3.0, -0.6249999999609375, 16.0, 10.0]

✅ Feature extractor ready! (18 features)


## Cell 4: Extract Features from All 50k+ Samples
**Kya karta hai:** Poore dataset pe `extract_features_v2()` run karta hai.

### Process:
1. Har row ka text lo
2. 18 features extract karo
3. NumPy matrix banao: `(n_samples, 18)`
4. NaN/Inf values remove karo (mathematical errors)

### Runtime:
~50,000 samples × 18 features = **~5-8 minutes on CPU**

### Output:
- `X` → feature matrix shape `(n_samples, 18)`
- `y` → labels array `(n_samples,)` — 0=human, 1=AI

In [5]:
print("[1/6] 🔬 Extracting 18 features from all samples...")
print(f"  Processing {len(df):,} samples...")
print(f"  (Estimated time: 5-8 minutes)")

X = np.array([
    extract_features_v2(text)
    for text in tqdm(df['text'].tolist(), desc="  Features", ncols=70)
], dtype=np.float32)

y = df['label'].values

# NaN/Inf values check and remove
nan_mask = ~np.isfinite(X).all(axis=1)
if nan_mask.sum() > 0:
    print(f"\n  ⚠ Removing {nan_mask.sum()} samples with NaN/Inf features")
    X = X[~nan_mask]
    y = y[~nan_mask]
    df = df[~nan_mask].reset_index(drop=True)

print(f"\n✅ Feature matrix ready!")
print(f"  Shape: {X.shape}  (samples × features)")
print(f"  Human samples: {(y==0).sum():,}")
print(f"  AI samples:    {(y==1).sum():,}")
print(f"\n  Feature value ranges (sanity check):")
for i, fname in enumerate(FEATURE_NAMES[:5]):
    print(f"    {fname:25s}: min={X[:,i].min():.3f}, max={X[:,i].max():.3f}, mean={X[:,i].mean():.3f}")


[1/6] 🔬 Extracting 18 features from all samples...
  Processing 50,160 samples...
  (Estimated time: 5-8 minutes)


  Features: 100%|█████████████| 50160/50160 [00:35<00:00, 1409.74it/s]



✅ Feature matrix ready!
  Shape: (50160, 18)  (samples × features)
  Human samples: 25,080
  AI samples:    25,080

  Feature value ranges (sanity check):
    avg_sent_len             : min=0.000, max=323.000, mean=20.629
    sent_len_std             : min=0.000, max=271.767, mean=7.445
    burstiness               : min=-1.000, max=0.406, mean=-0.479
    max_sent_len             : min=0.000, max=582.000, mean=34.660
    min_sent_len             : min=0.000, max=323.000, mean=10.795


## Cell 5: Train-Test Split + Feature Scaling
**Kya karta hai:** Data ko train/test mein split karo aur features scale karo.

### Kyun Scale?
- Features ke ranges bahut different hain:
  - `avg_sent_len` → 5 to 50
  - `comma_ratio` → 0.001 to 0.05
- **StandardScaler** → mean=0, std=1 karta hai
- Logistic Regression ke liye **zaroori** hai
- Random Forest ke liye helpful hai (slightly)

### Train-Test Split:
- **85% training** — model seekhta hai
- **15% testing** — model kitna accurate hai
- `stratify=y` → train aur test mein same human/AI ratio

### Class Weights:
Agar 60k human aur 40k AI hain → model human ke taraf biased ho sakta hai.
`class_weight` se yeh fix hota hai.

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("[2/6] ⚖ Train-Test Split + Feature Scaling...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.15,      # 15% test set
    random_state=42,     # Reproducibility ke liye fixed seed
    stratify=y           # Same ratio in train and test
)

print(f"  Train set: {len(X_train):,} samples")
print(f"  Test set:  {len(X_test):,} samples")
print(f"  Train Human/AI: {(y_train==0).sum():,} / {(y_train==1).sum():,}")
print(f"  Test  Human/AI: {(y_test==0).sum():,}  / {(y_test==1).sum():,}")

# ── Feature Scaling ───────────────────────────────────────────────────────────
# StandardScaler: (x - mean) / std — sab features same scale pe aate hain
scaler    = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # Train pe fit karo
X_test_s  = scaler.transform(X_test)        # Test pe sirf transform (fit mat karo!)

# Scaler save karo — inference time pe bhi use hoga
joblib.dump(scaler, os.path.join(MODELS_DIR, 'feature_scaler.pkl'))
print(f"\n  ✅ StandardScaler fitted and saved")

# ── Class weights calculate karo ───────────────────────────────────────────────
n_human = (y_train == 0).sum()
n_ai    = (y_train == 1).sum()
cw      = {0: 1.0, 1: n_human / (n_ai + 1e-9)}
print(f"  Class weights: human=1.0, ai={cw[1]:.2f}")
if cw[1] > 1.5:
    print(f"  ⚠ AI samples less hain — upweighting by {cw[1]:.1f}x")


[2/6] ⚖ Train-Test Split + Feature Scaling...
  Train set: 42,636 samples
  Test set:  7,524 samples
  Train Human/AI: 21,318 / 21,318
  Test  Human/AI: 3,762  / 3,762

  ✅ StandardScaler fitted and saved
  Class weights: human=1.0, ai=1.00


## Cell 6: Train 3 Individual Models
**Kya karta hai:** Teen alag-alag ML models train karta hai — baad mein inhe ensemble banayenge.

### Model 1: Random Forest (300 trees)
- Bahut interpretable — feature importance dikhata hai
- Fast predict — good for production
- `n_jobs=-1` → saare CPU cores use karo

### Model 2: Gradient Boosting (300 estimators)
- Tabular data pe best performing algorithm
- Slower to train but very accurate
- `learning_rate=0.05` — chote steps mein seekhta hai (better generalization)

### Model 3: Logistic Regression
- Fastest, most interpretable
- Good calibrated probabilities
- Linear model — complementary to tree-based models

### Kyun Teen Models?
Har model alag tarah se learn karta hai. Ensemble mein milana = kam errors.

In [7]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

print("[3/6] 🤖 Training 3 Individual Models...")

# ── Model 1: Random Forest ─────────────────────────────────────────────────────
print("\n  [1/3] Random Forest (300 trees)...")
print("        n_jobs=-1 → all CPU cores use karega")
rf = RandomForestClassifier(
    n_estimators=300,       # Trees ka count (zyada = better, slower)
    max_depth=None,         # Unlimited depth — tree fully grow karo
    min_samples_split=5,    # Node split ke liye min 5 samples
    min_samples_leaf=2,     # Leaf mein min 2 samples (overfitting reduce)
    max_features='sqrt',    # √features randomly consider (diversity)
    class_weight=cw,        # Imbalance handle
    n_jobs=-1,              # All CPU cores
    random_state=42,
)
rf.fit(X_train_s, y_train)
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test_s)[:,1])
rf_acc = accuracy_score(y_test, rf.predict(X_test_s))
print(f"        ✓ AUC: {rf_auc:.4f} | Accuracy: {rf_acc:.4f}")

# ── Model 2: Gradient Boosting ────────────────────────────────────────────────
print("\n  [2/3] Gradient Boosting (300 estimators)...")
print("        Ye slow hai — 5-10 minutes lagenge, wait karo!")
gb = GradientBoostingClassifier(
    n_estimators=300,       # Boosting rounds
    learning_rate=0.05,     # Chote steps = better generalization
    max_depth=5,            # Tree depth per round
    min_samples_split=5,
    subsample=0.8,          # 80% samples per round (variance reduce)
    max_features='sqrt',    # Feature subsampling
    random_state=42,
)
gb.fit(X_train_s, y_train)
gb_auc = roc_auc_score(y_test, gb.predict_proba(X_test_s)[:,1])
gb_acc = accuracy_score(y_test, gb.predict(X_test_s))
print(f"        ✓ AUC: {gb_auc:.4f} | Accuracy: {gb_acc:.4f}")

# ── Model 3: Logistic Regression ──────────────────────────────────────────────
print("\n  [3/3] Logistic Regression (fast)...")
lr = LogisticRegression(
    C=1.0,                  # Regularization (1.0 = default, balanced)
    class_weight=cw,        # Imbalance handle
    max_iter=1000,          # Convergence ke liye enough iterations
    random_state=42,
    n_jobs=-1,
)
lr.fit(X_train_s, y_train)
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_s)[:,1])
lr_acc = accuracy_score(y_test, lr.predict(X_test_s))
print(f"        ✓ AUC: {lr_auc:.4f} | Accuracy: {lr_acc:.4f}")

print(f"\n  Individual model comparison:")
print(f"    Random Forest:      AUC={rf_auc:.4f}  Acc={rf_acc:.4f}")
print(f"    Gradient Boosting:  AUC={gb_auc:.4f}  Acc={gb_acc:.4f}")
print(f"    Logistic Reg:       AUC={lr_auc:.4f}  Acc={lr_acc:.4f}")


[3/6] 🤖 Training 3 Individual Models...

  [1/3] Random Forest (300 trees)...
        n_jobs=-1 → all CPU cores use karega
        ✓ AUC: 0.9364 | Accuracy: 0.8600

  [2/3] Gradient Boosting (300 estimators)...
        Ye slow hai — 5-10 minutes lagenge, wait karo!
        ✓ AUC: 0.9429 | Accuracy: 0.8646

  [3/3] Logistic Regression (fast)...
        ✓ AUC: 0.8988 | Accuracy: 0.8235

  Individual model comparison:
    Random Forest:      AUC=0.9364  Acc=0.8600
    Gradient Boosting:  AUC=0.9429  Acc=0.8646
    Logistic Reg:       AUC=0.8988  Acc=0.8235


## Cell 7: Ensemble + Calibration
**Kya karta hai:** Teen models ko combine karke ek powerful ensemble banata hai, phir probabilities calibrate karta hai.

### VotingClassifier (Soft Voting):
- Har model ka probability output lete hain
- **Weighted average** karte hain — better model ko zyada weight
- Better than any single model

### Weight Calculation:
```
AUC scores ke proportional weights:
  RF weight   = rf_auc / (rf_auc + gb_auc + lr_auc)
  GB weight   = gb_auc / (rf_auc + gb_auc + lr_auc)
  LR weight   = lr_auc / (rf_auc + gb_auc + lr_auc)
```
Matlab jo model better perform kare, uski baat zyada maano.

### Calibration (Platt Scaling):
Raw model probabilities often **overconfident** hote hain:
- Model kehta hai 95% AI → actually 78% ho sakta hai
- CalibratedClassifierCV yeh fix karta hai
- `method='sigmoid'` = Platt scaling

In [8]:
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV

print("[4/6] 🔗 Building Voting Ensemble + Calibration...")

# ── Weighted Soft Voting ───────────────────────────────────────────────────────
# AUC scores ke proportional weights assign karo
total_w = rf_auc + gb_auc + lr_auc
w_rf = rf_auc / total_w
w_gb = gb_auc / total_w
w_lr = lr_auc / total_w

print(f"\n  Ensemble weights (AUC-proportional):")
print(f"    Random Forest:     {w_rf:.3f}")
print(f"    Gradient Boosting: {w_gb:.3f}")
print(f"    Logistic Reg:      {w_lr:.3f}")

ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('gb', gb),
        ('lr', lr),
    ],
    voting='soft',           # Probability average (not just vote count)
    weights=[w_rf, w_gb, w_lr],
    n_jobs=-1,
)
ensemble.fit(X_train_s, y_train)
ens_auc = roc_auc_score(y_test, ensemble.predict_proba(X_test_s)[:,1])
ens_acc = accuracy_score(y_test, ensemble.predict(X_test_s))
print(f"\n  Voting Ensemble: AUC={ens_auc:.4f} | Acc={ens_acc:.4f}")

# ── Probability Calibration ────────────────────────────────────────────────────
print("\n  Calibrating probabilities (Platt scaling, cv=3)...")
print("  (Yeh 3-fold CV karta hai — 2-3 minutes)")
calibrated = CalibratedClassifierCV(
    ensemble,
    method='sigmoid',   # Platt scaling — reliable for binary classification
    cv=3                # 3-fold cross-validation
)
calibrated.fit(X_train_s, y_train)
cal_auc = roc_auc_score(y_test, calibrated.predict_proba(X_test_s)[:,1])
cal_acc = accuracy_score(y_test, calibrated.predict(X_test_s))
print(f"  Calibrated Ensemble: AUC={cal_auc:.4f} | Acc={cal_acc:.4f}")

# Improvement check
improvement = cal_auc - max(rf_auc, gb_auc, lr_auc)
print(f"\n  AUC improvement over best single model: +{improvement:.4f}")


[4/6] 🔗 Building Voting Ensemble + Calibration...

  Ensemble weights (AUC-proportional):
    Random Forest:     0.337
    Gradient Boosting: 0.339
    Logistic Reg:      0.324

  Voting Ensemble: AUC=0.9359 | Acc=0.8587

  Calibrating probabilities (Platt scaling, cv=3)...
  (Yeh 3-fold CV karta hai — 2-3 minutes)
  Calibrated Ensemble: AUC=0.9356 | Acc=0.8573

  AUC improvement over best single model: +-0.0072


## Cell 8: Comprehensive Evaluation + Plots
**Kya karta hai:** Saare models ko evaluate karta hai aur 4 plots banata hai.

### Plots:
1. **Model Comparison** — AUC vs Accuracy bar chart
2. **Confusion Matrix** — True/False Positive/Negative breakdown
3. **ROC Curves** — All models ka true/false positive tradeoff
4. **Feature Importance** — Konse features zyada important hain

### Key Metrics:
- **AUC (Area Under ROC Curve)** → 1.0 = perfect, 0.5 = random
- **Accuracy** → % correct predictions
- **F1 Score** → Precision + Recall ka harmonic mean (good for imbalanced data)

In [12]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, average_precision_score
)

print("[5/6] 📊 Comprehensive Evaluation...")

models = {
    'Random Forest':       (rf,         rf_auc),
    'Grad Boosting':       (gb,         gb_auc),
    'Logistic Reg':        (lr,         lr_auc),
    'Voting Ensemble':     (ensemble,   ens_auc),
    'Calibrated Ensemble': (calibrated, cal_auc),
}

results = {}
for name, (model, auc) in models.items():
    y_pred  = model.predict(X_test_s)
    y_proba = model.predict_proba(X_test_s)[:,1]
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'f1':       f1_score(y_test, y_pred, average='weighted'),
        'auc':      auc,
    }

print("\n  Model Comparison:")
print(f"  {'Model':25s} | {'Accuracy':>10} | {'F1':>8} | {'AUC':>8}")
print("  " + "-"*57)
for name, r in results.items():
    marker = " ← BEST" if r['auc'] == max(v['auc'] for v in results.values()) else ""
    print(f"  {name:25s} | {r['accuracy']:>10.4f} | {r['f1']:>8.4f} | {r['auc']:>8.4f}{marker}")

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('AI Detection Model Evaluation v2 — IntegriCheck', fontsize=14, fontweight='bold')

# 1. Model comparison bar chart
ax = axes[0, 0]
model_names = list(results.keys())
aucs = [results[m]['auc'] for m in model_names]
accs = [results[m]['accuracy'] for m in model_names]
x    = np.arange(len(model_names))
ax.bar(x - 0.2, aucs, 0.35, label='ROC-AUC', color='#2196F3', alpha=0.85)
ax.bar(x + 0.2, accs, 0.35, label='Accuracy', color='#4CAF50', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([m.split()[-1] for m in model_names], rotation=20, ha='right', fontsize=8)
ax.set_ylim(0.70, 1.01)
ax.set_title('Model Comparison')
ax.legend(fontsize=9)
ax.set_ylabel('Score')
for i, (bar_a, bar_b) in enumerate(zip(ax.patches[:5], ax.patches[5:])):
    ax.text(bar_a.get_x()+bar_a.get_width()/2, bar_a.get_height()+0.003,
            f'{bar_a.get_height():.3f}', ha='center', va='bottom', fontsize=7)

# 2. Confusion matrix (best model)
ax = axes[0, 1]
best_name  = max(results, key=lambda k: results[k]['auc'])
best_model = models[best_name][0]
y_pred_cm  = best_model.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred_cm)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Human','AI'], yticklabels=['Human','AI'],
            annot_kws={'size': 12})
ax.set_title(f'Confusion Matrix ({best_name})')
ax.set_ylabel('True Label', fontsize=9)
ax.set_xlabel('Predicted Label', fontsize=9)
tn, fp, fn, tp = cm.ravel()
ax.set_xlabel(f'Predicted | TN:{tn} FP:{fp} FN:{fn} TP:{tp}', fontsize=8)

# 3. ROC Curves
ax = axes[1, 0]
colors_list = ['#e74c3c','#3498db','#27ae60','#f39c12','#9b59b6']
for (name, (model, _)), color in zip(models.items(), colors_list):
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_test_s)[:,1])
    lw = 2.5 if 'Calibrated' in name else 1.2
    ax.plot(fpr, tpr, label=f"{name.split()[-1]} ({results[name]['auc']:.3f})",
            color=color, linewidth=lw)
ax.plot([0,1],[0,1],'k--', linewidth=0.8, label='Random (0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 4. Feature importance (from RF)
ax = axes[1, 1]
fi         = rf.feature_importances_
sorted_idx = fi.argsort()[::-1]
colors_fi  = ['#e74c3c' if fi[i] > np.percentile(fi, 75) else '#3498db' for i in sorted_idx]
ax.barh(range(len(sorted_idx)), fi[sorted_idx], color=colors_fi, alpha=0.8)
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([FEATURE_NAMES[i] for i in sorted_idx], fontsize=8)
ax.set_title('Feature Importances (Random Forest)')
ax.set_xlabel('Importance Score')
ax.invert_yaxis()
# Annotate top features
for i, (idx, imp) in enumerate(zip(sorted_idx[:3], fi[sorted_idx[:3]])):
    ax.text(imp + 0.001, i, f'{imp:.3f}', va='center', fontsize=7, color='#e74c3c')

plt.tight_layout()
eval_path = os.path.join(PROCESSED_DIR, 'ai_detection_evaluation_v2.png')
plt.savefig(eval_path, dpi=130, bbox_inches='tight')
plt.show()

print(f"\n  Plot saved: {eval_path}")
print(f"\n  🏆 Best Model: {best_name}")
print(f"     AUC: {results[best_name]['auc']:.4f} | Accuracy: {results[best_name]['accuracy']:.4f}")


[5/6] 📊 Comprehensive Evaluation...

  Model Comparison:
  Model                     |   Accuracy |       F1 |      AUC
  ---------------------------------------------------------
  Random Forest             |     0.8600 |   0.8600 |   0.9364
  Grad Boosting             |     0.8646 |   0.8645 |   0.9429 ← BEST
  Logistic Reg              |     0.8235 |   0.8235 |   0.8988
  Voting Ensemble           |     0.8587 |   0.8587 |   0.9359
  Calibrated Ensemble       |     0.8573 |   0.8572 |   0.9356

  Plot saved: e:\ADD\integricheck_project\data\processed\ai_detection_evaluation_v2.png

  🏆 Best Model: Grad Boosting
     AUC: 0.9429 | Accuracy: 0.8646


## Cell 9: Save Models + Feature Config
**Kya karta hai:** Best model aur saari config save karta hai.

### Files Saved:
1. `ai_detection_model.pkl` — **Best model** (engine.py use karta hai)
2. `all_ai_models.pkl` — Saare 5 models (future reference)
3. `feature_scaler.pkl` — StandardScaler (already saved in Cell 5)
4. `feature_extractor_config.json` — Metadata + accuracy scores

### Kyun JSON config?
Engine.py runtime pe yeh config load karta hai → pata chalta hai kitne features expect karne hain (12 vs 18). Isse backward compatibility maintain hoti hai.

In [13]:
print("[6/6] 💾 Saving Models + Config...")

# ── Best model save karo (yahi engine.py use karta hai) ──────────────────────
joblib.dump(best_model,
            os.path.join(MODELS_DIR, 'ai_detection_model.pkl'),
            compress=3)
print(f"  ✅ Best model saved: {best_name}")

# ── Saare models save karo (future use ke liye) ───────────────────────────────
all_models_dict = {
    'random_forest':       rf,
    'gradient_boosting':   gb,
    'logistic_regression': lr,
    'voting_ensemble':     ensemble,
    'calibrated_ensemble': calibrated,
    'best_model_name':     best_name,
}
joblib.dump(all_models_dict,
            os.path.join(MODELS_DIR, 'all_ai_models.pkl'),
            compress=3)
print(f"  ✅ All 5 models saved")

# ── Feature extractor config save karo ───────────────────────────────────────
config = {
    'best_model_name': best_name,
    'feature_names':   FEATURE_NAMES,
    'n_features':      len(FEATURE_NAMES),         # 18 (engine.py check karta hai)
    'accuracy':        results[best_name]['accuracy'],
    'f1_score':        results[best_name]['f1'],
    'auc':             results[best_name]['auc'],
    'trained_on':      datetime.now().isoformat(),
    'train_samples':   len(X_train),
    'test_samples':    len(X_test),
    'all_model_aucs': {name: results[name]['auc'] for name in results},
}

with open(os.path.join(MODELS_DIR, 'feature_extractor_config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print(f"  ✅ Feature config saved (n_features=18)")

# ── Show saved files ──────────────────────────────────────────────────────────
print(f"\n  Saved files:")
for fname in ['ai_detection_model.pkl', 'all_ai_models.pkl',
              'feature_scaler.pkl', 'feature_extractor_config.json']:
    path = os.path.join(MODELS_DIR, fname)
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"    ✅ {fname:40s} {size:.1f} MB")


[6/6] 💾 Saving Models + Config...
  ✅ Best model saved: Grad Boosting
  ✅ All 5 models saved
  ✅ Feature config saved (n_features=18)

  Saved files:
    ✅ ai_detection_model.pkl                   0.5 MB
    ✅ all_ai_models.pkl                        163.0 MB
    ✅ feature_scaler.pkl                       0.0 MB
    ✅ feature_extractor_config.json            0.0 MB


## Cell 10: Update engine.py + Final Test
**Kya karta hai:** Naya engine.py copy karta hai (18 features support karta hai) aur ek final real-world test karta hai.

### ⚠️ Important:
`engine_v2.py` ko `src/ai_detection/engine.py` pe copy karna zaroori hai. Purana engine 12 features expect karta tha — naya model 18 expect karta hai. Agar engine update nahi kiya toh mismatch error aayegi.

### Real-world Test:
3 texts test karo:
1. **Human student essay** — should be LOW AI probability
2. **AI-generated text** (GPTisms) — should be HIGH AI probability  
3. **Mixed text** — should be MEDIUM AI probability

In [15]:
print("[FINAL] 🔄 Updating engine.py + Testing...")

# ── Engine.py update karo ─────────────────────────────────────────────────────
import shutil

engine_v2_candidates = [
    os.path.join(BASE_DIR, 'engine_v2.py'),
    os.path.join(BASE_DIR, '..', 'engine_v2.py'),
    'engine_v2.py',
]

engine_dest = os.path.join(BASE_DIR, 'src', 'ai_detection', 'engine.py')
engine_updated = False

for src_path in engine_v2_candidates:
    if os.path.exists(src_path):
        shutil.copy2(src_path, engine_dest)
        print(f"  ✅ engine.py updated from: {src_path}")
        engine_updated = True
        break

if not engine_updated:
    print(f"  ⚠ engine_v2.py nahi mila — manually copy karo:")
    print(f"     cp engine_v2.py {engine_dest}")

# ── Fresh import ───────────────────────────────────────────────────────────────
sys.path.insert(0, BASE_DIR)
for mod_name in list(sys.modules.keys()):
    if 'ai_detection' in mod_name or 'engine' in mod_name:
        del sys.modules[mod_name]

from src.ai_detection.engine import analyze_ai_detection

# ── Real-world tests ───────────────────────────────────────────────────────────
test_human = (
    "I struggled a lot with this assignment at first. I kept making silly mistakes "
    "in my calculations. My professor told me to re-read chapter 5, so I did and "
    "finally understood what regression actually means. It's not that scary now!"
)

test_ai = (
    "Furthermore, it is important to note that machine learning leverages robust "
    "algorithms to seamlessly process multifaceted datasets. Moreover, this paradigm "
    "holistically embodies the intricate nuances of data-driven decision making. "
    "In conclusion, the pivotal role of AI in fostering innovation cannot be understated."
)

test_mixed = (
    "Machine learning is a useful tool for data analysis. I used Python and scikit-learn "
    "for my project. However, it is important to note that data preprocessing is crucial "
    "before training any model. The results showed 85% accuracy on the test set."
)

print("\n  Real-world detection tests:")
print("  " + "-"*55)

for label, text in [("Human essay", test_human), ("AI-generated", test_ai), ("Mixed", test_mixed)]:
    result = analyze_ai_detection(text)
    ai_pct = result.get('ai_probability', 0)
    verdict = result.get('verdict', '')
    n_feat  = result.get('n_features', '?')
    print(f"\n  [{label}]")
    print(f"    AI Probability: {ai_pct:.1f}% | {verdict}")
    print(f"    Features used:  {n_feat}")

print("\n  " + "="*55)
print("  STEP 4 COMPLETE ✅")
print("  " + "="*55)
print(f"\n  Final Model Performance:")
print(f"    Best Model:  {best_name}")
print(f"    Accuracy:    {results[best_name]['accuracy']*100:.1f}%")
print(f"    ROC-AUC:     {results[best_name]['auc']:.4f}")
improvement = results[best_name]['accuracy'] - 0.8891
print(f"    Improvement: +{improvement*100:.1f}% over baseline (88.9%)")
print(f"\n  ▶ Next: python run_app.py → open http://localhost:5000")
print("  " + "="*55)


[FINAL] 🔄 Updating engine.py + Testing...
  ✅ engine.py updated from: e:\ADD\integricheck_project\..\engine_v2.py

  Real-world detection tests:
  -------------------------------------------------------

  [Human essay]
    AI Probability: 56.0% | POSSIBLY AI GENERATED
    Features used:  18

  [AI-generated]
    AI Probability: 90.7% | LIKELY AI GENERATED
    Features used:  18

  [Mixed]
    AI Probability: 87.2% | LIKELY AI GENERATED
    Features used:  18

  STEP 4 COMPLETE ✅

  Final Model Performance:
    Best Model:  Grad Boosting
    Accuracy:    86.5%
    ROC-AUC:     0.9429
    Improvement: +-2.5% over baseline (88.9%)

  ▶ Next: python run_app.py → open http://localhost:5000
